
# SIRV-style RNA Velocity, Xenium Spatial Integration, H&E Overlay, and PAGA

This notebook implements a **GitHub-ready workflow** for combining:

- a **single-cell RNA-seq reference** with spliced and unspliced counts from **Velocyto loom files**
- **scVelo dynamical RNA velocity**
- **Xenium spatial transcriptomics**
- **H&E image overlay**
- **standard PAGA graph abstraction**

The notebook **does not use manual PAGA arrows**. Instead, it relies on the model-derived outputs from scVelo/Scanpy and a reference-to-spatial transfer step.

## Inputs
- Reference scRNA-seq object (`.h5ad`)
- Velocyto loom file for the same reference cells
- Xenium spatial object (`.h5ad`) or equivalent AnnData
- Optional ROI cell list
- Optional annotation file for Xenium labels
- Optional H&E image (`.ome.tif`, `.tif`, or `.png`)

## Outputs
- Updated reference AnnData with RNA velocity, velocity confidence, velocity pseudotime, and PAGA
- Updated Xenium AnnData with transferred labels and transferred velocity-derived metrics
- Spatial plots, H&E overlays, and standard PAGA figures


In [ ]:

# ============================================================
# 0) Imports
# ============================================================
import os
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import anndata as ad
import matplotlib.pyplot as plt
import scipy.sparse as sp

from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from scipy.stats import mode

try:
    import tifffile
except ImportError:
    tifffile = None

warnings.filterwarnings("ignore")
scv.settings.verbosity = 3
scv.settings.presenter_view = True
sc.set_figure_params(figsize=(7, 7), dpi=120)



## 1. User settings

Edit only this section before running the notebook.


In [ ]:

# ============================================================
# 1) User settings
# ============================================================
project_name = "BTD10_Xenium_SIRV_workflow"

# Reference scRNA-seq files
reference_h5ad_path = "/path/to/BTD10_reference.h5ad"
loom_path           = "/path/to/BTD10_velocyto.loom"

# Xenium spatial object
xenium_h5ad_path    = "/path/to/Xenium_query.h5ad"

# Optional: ROI file listing a subset of Xenium cells
roi_stats_csv_path  = "/path/to/6263_RO1_Selection_1_cells_stats.csv"   # set to None if not used

# Optional: external annotation file with columns like cell_id / group
use_for_xenium_csv  = "/path/to/use for xenium.csv"                      # set to None if not used

# Optional H&E image
he_image_path       = "/path/to/6263_H&E_FIXED.ome.tif"                  # set to None if not used

# Optional: columns already present in Xenium object for H&E coordinates
he_x_col = "x_he"   # fallback handled below if not present
he_y_col = "y_he"

# Metadata columns
reference_label_col = "Subtype_Refined"
transferred_label_col = "Celltype_from_use_for_xenium"

# General settings
outdir = "/path/to/output_directory"
os.makedirs(outdir, exist_ok=True)

n_pcs = 30
n_neighbors = 15
min_shared_genes = 300

# PAGA grouping column for the spatial object
spatial_group_col = transferred_label_col

print("Output directory:", outdir)


## 2. Helper functions

In [ ]:

# ============================================================
# 2) Helper functions
# ============================================================
def read_csv_flex(path):
    if path is None:
        return None
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.read_csv(path, sep="\t")


def ensure_cell_id_columns(adata):
    if "cell_id_raw" not in adata.obs.columns:
        adata.obs["cell_id_raw"] = adata.obs_names.astype(str)
    if "cell_id_clean" not in adata.obs.columns:
        adata.obs["cell_id_clean"] = (
            pd.Series(adata.obs["cell_id_raw"].astype(str), index=adata.obs.index)
            .str.replace("-1$", "", regex=True)
            .str.strip()
        )
    adata.obs["cell_id_raw"] = adata.obs["cell_id_raw"].astype(str).str.strip()
    adata.obs["cell_id_clean"] = adata.obs["cell_id_clean"].astype(str).str.strip()
    return adata


def ensure_spatial_columns(adata):
    if "x_centroid" in adata.obs.columns and "y_centroid" in adata.obs.columns:
        adata.obs["x_centroid"] = pd.to_numeric(adata.obs["x_centroid"], errors="coerce")
        adata.obs["y_centroid"] = pd.to_numeric(adata.obs["y_centroid"], errors="coerce")
        adata.obsm["spatial"] = adata.obs[["x_centroid", "y_centroid"]].to_numpy()
    elif "spatial" in adata.obsm.keys():
        adata.obs["x_centroid"] = adata.obsm["spatial"][:, 0]
        adata.obs["y_centroid"] = adata.obsm["spatial"][:, 1]
    else:
        raise ValueError("No spatial coordinates found. Provide x_centroid/y_centroid or obsm['spatial'].")
    return adata


def make_unique_gene_names_upper(adata):
    adata.var_names = pd.Index([str(x).upper() for x in adata.var_names])
    adata.var_names_make_unique()
    return adata


def preprocess_for_transfer(adata):
    tmp = adata.copy()
    sc.pp.normalize_total(tmp, target_sum=1e4)
    sc.pp.log1p(tmp)
    return tmp


def sparse_to_array(x):
    if sp.issparse(x):
        return x.toarray()
    return np.asarray(x)


def transfer_reference_to_query(
    ref_adata,
    query_adata,
    label_col="Subtype_Refined",
    metrics=("velocity_pseudotime", "velocity_confidence", "velocity_length"),
    n_pcs=30,
    k=15
):
    '''
    Transfer labels and velocity-derived metrics from reference to query
    using shared genes and k-nearest neighbors in PCA space.
    '''
    ref = preprocess_for_transfer(ref_adata)
    qry = preprocess_for_transfer(query_adata)

    shared = ref.var_names.intersection(qry.var_names)
    if len(shared) < min_shared_genes:
        raise ValueError(f"Too few shared genes ({len(shared)}).")

    ref = ref[:, shared].copy()
    qry = qry[:, shared].copy()

    X_ref = sparse_to_array(ref.X)
    X_qry = sparse_to_array(qry.X)

    pca = PCA(n_components=min(n_pcs, X_ref.shape[1], X_ref.shape[0] - 1), random_state=0)
    ref_pca = pca.fit_transform(X_ref)
    qry_pca = pca.transform(X_qry)

    nbrs = NearestNeighbors(n_neighbors=min(k, ref_pca.shape[0]), metric="euclidean")
    nbrs.fit(ref_pca)
    dists, idxs = nbrs.kneighbors(qry_pca)

    # inverse-distance weights
    weights = 1.0 / (dists + 1e-8)
    weights = weights / weights.sum(axis=1, keepdims=True)

    # label transfer by weighted vote
    ref_labels = ref.obs[label_col].astype(str).values
    transferred_labels = []
    for i in range(idxs.shape[0]):
        neigh_labels = ref_labels[idxs[i]]
        # weighted score per label
        score = {}
        for lab, w in zip(neigh_labels, weights[i]):
            score[lab] = score.get(lab, 0) + w
        transferred_labels.append(max(score, key=score.get))

    query_adata.obs[label_col + "_transfer"] = transferred_labels
    query_adata.obsm["X_ref_pca"] = qry_pca

    # continuous metrics by weighted mean
    for metric in metrics:
        if metric in ref.obs.columns:
            ref_vals = pd.to_numeric(ref.obs[metric], errors="coerce").values
            transferred_vals = np.sum(ref_vals[idxs] * weights, axis=1)
            query_adata.obs[metric] = transferred_vals

    # confidence score for label transfer
    max_vote = []
    for i in range(idxs.shape[0]):
        neigh_labels = ref_labels[idxs[i]]
        score = {}
        for lab, w in zip(neigh_labels, weights[i]):
            score[lab] = score.get(lab, 0) + w
        max_vote.append(max(score.values()))
    query_adata.obs["transfer_confidence"] = max_vote

    return query_adata


def load_he_image(path):
    if path is None:
        return None
    if not os.path.exists(path):
        raise FileNotFoundError(f"H&E image not found: {path}")
    ext = os.path.splitext(path)[1].lower()

    if ext in [".png", ".jpg", ".jpeg"]:
        import matplotlib.image as mpimg
        img = mpimg.imread(path)
        return img

    if tifffile is None:
        raise ImportError("tifffile is required to read TIFF/OME-TIFF files.")

    img = tifffile.imread(path)
    img = np.squeeze(img)

    # Handle common OME-TIFF layouts
    if img.ndim == 3 and img.shape[0] in [3, 4] and img.shape[-1] not in [3, 4]:
        img = np.moveaxis(img, 0, -1)

    # If grayscale, convert by repeating channel
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)

    return img


def choose_overlay_coords(adata, preferred_x="x_he", preferred_y="y_he"):
    if preferred_x in adata.obs.columns and preferred_y in adata.obs.columns:
        x = pd.to_numeric(adata.obs[preferred_x], errors="coerce")
        y = pd.to_numeric(adata.obs[preferred_y], errors="coerce")
        return x, y, preferred_x, preferred_y

    if "x_centroid" in adata.obs.columns and "y_centroid" in adata.obs.columns:
        x = pd.to_numeric(adata.obs["x_centroid"], errors="coerce")
        y = pd.to_numeric(adata.obs["y_centroid"], errors="coerce")
        return x, y, "x_centroid", "y_centroid"

    raise ValueError("No coordinate columns available for H&E overlay.")


def plot_he_overlay(adata, color_col, image_path, outpath, preferred_x="x_he", preferred_y="y_he", point_size=10):
    img = load_he_image(image_path)
    x, y, xcol, ycol = choose_overlay_coords(adata, preferred_x, preferred_y)

    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(img)

    vals = adata.obs[color_col]
    if pd.api.types.is_numeric_dtype(vals):
        sc = ax.scatter(x, y, c=vals, s=point_size, cmap="viridis", alpha=0.8, linewidths=0)
        plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02, label=color_col)
    else:
        categories = vals.astype(str).unique().tolist()
        palette = {c: col for c, col in zip(categories, sc.pl.palettes.default_102[:len(categories)])}
        for cat in categories:
            mask = vals.astype(str) == cat
            ax.scatter(x[mask], y[mask], s=point_size, c=palette[cat], alpha=0.8, linewidths=0, label=cat)
        ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=True, title=color_col)

    ax.set_title(f"{color_col} over H&E")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(outpath, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()


## 3. Load reference scRNA-seq data and Velocyto loom

In [ ]:

# ============================================================
# 3) Load reference and loom
# ============================================================
ref = sc.read_h5ad(reference_h5ad_path)
ref = ensure_cell_id_columns(ref)
ref = make_unique_gene_names_upper(ref)

print(ref)
print("Reference obs columns:", ref.obs.columns.tolist()[:20])

loom = scv.read(loom_path, cache=True)
loom = ensure_cell_id_columns(loom)
loom = make_unique_gene_names_upper(loom)

print(loom)


In [ ]:

# ============================================================
# 4) Harmonize barcodes and merge loom layers into reference
# ============================================================
# Velocyto barcodes often differ in suffix formatting
ref_barcodes = pd.Index(ref.obs["cell_id_clean"].astype(str))
loom_barcodes = pd.Index(loom.obs["cell_id_clean"].astype(str))

common_cells = ref_barcodes.intersection(loom_barcodes)
if len(common_cells) == 0:
    raise ValueError("No overlapping cells between reference h5ad and loom file after barcode cleaning.")

ref_idx = ref.obs["cell_id_clean"].isin(common_cells)
loom_idx = loom.obs["cell_id_clean"].isin(common_cells)

ref_sub = ref[ref_idx].copy()
loom_sub = loom[loom_idx].copy()

# Reorder loom to match reference
order = pd.Index(loom_sub.obs["cell_id_clean"]).get_indexer(ref_sub.obs["cell_id_clean"])
loom_sub = loom_sub[order].copy()

# Restrict to shared genes
shared_genes = ref_sub.var_names.intersection(loom_sub.var_names)
if len(shared_genes) == 0:
    raise ValueError("No overlapping genes between reference h5ad and loom file.")

ref_sub = ref_sub[:, shared_genes].copy()
loom_sub = loom_sub[:, shared_genes].copy()

# Attach spliced/unspliced layers from loom
for layer_name in ["spliced", "unspliced", "ambiguous"]:
    if layer_name in loom_sub.layers:
        ref_sub.layers[layer_name] = loom_sub.layers[layer_name].copy()

# Carry forward raw counts if needed
if "spliced" not in ref_sub.layers or "unspliced" not in ref_sub.layers:
    raise ValueError("Merged reference is missing spliced/unspliced layers after loom integration.")

adata_ref = ref_sub.copy()
print(adata_ref)
print("Layers:", list(adata_ref.layers.keys()))


## 4. scVelo dynamical RNA velocity on the reference

In [ ]:

# ============================================================
# 5) Preprocess reference for scVelo
# ============================================================
scv.pp.filter_and_normalize(adata_ref, min_shared_counts=20, n_top_genes=3000)
scv.pp.moments(adata_ref, n_pcs=min(n_pcs, 30), n_neighbors=n_neighbors)

# Keep label column as categorical if present
if reference_label_col in adata_ref.obs.columns:
    adata_ref.obs[reference_label_col] = adata_ref.obs[reference_label_col].astype("category")

# Compute a UMAP if not already present
if "X_umap" not in adata_ref.obsm.keys():
    sc.tl.umap(adata_ref)

print(adata_ref)


In [ ]:

# ============================================================
# 6) Dynamical model, velocity, confidence, pseudotime, terminal states
# ============================================================
scv.tl.recover_dynamics(adata_ref)
scv.tl.velocity(adata_ref, mode="dynamical")
scv.tl.velocity_graph(adata_ref)
scv.tl.velocity_confidence(adata_ref)
scv.tl.velocity_pseudotime(adata_ref)
scv.tl.terminal_states(adata_ref)

adata_ref.write(os.path.join(outdir, "BTD10_reference_velocity.h5ad"))
print("Saved:", os.path.join(outdir, "BTD10_reference_velocity.h5ad"))


In [ ]:

# ============================================================
# 7) Velocity plots on the reference
# ============================================================
scv.pl.velocity_embedding_stream(adata_ref, basis="umap", color=reference_label_col, save=None)
scv.pl.scatter(adata_ref, color="velocity_pseudotime", color_map="plasma")
scv.pl.scatter(adata_ref, color="velocity_confidence", color_map="viridis")
scv.pl.scatter(adata_ref, color="root_cells", color_map="magma")
scv.pl.scatter(adata_ref, color="end_points", color_map="magma")


## 5. Standard PAGA on the reference (no manual arrows)

In [ ]:

# ============================================================
# 8) Standard PAGA on reference groups
# ============================================================
if reference_label_col not in adata_ref.obs.columns:
    raise ValueError(f"{reference_label_col} is not present in the reference object.")

sc.pp.neighbors(adata_ref, n_neighbors=n_neighbors, use_rep="X_pca" if "X_pca" in adata_ref.obsm else None)
sc.tl.paga(adata_ref, groups=reference_label_col)

sc.pl.paga(adata_ref, color=reference_label_col, threshold=0.03)
sc.pl.paga_compare(
    adata_ref,
    basis="umap",
    color=reference_label_col,
    threshold=0.03,
    right_margin=0.2,
    size=20
)

# Save PAGA connectivity matrix
if "paga" in adata_ref.uns:
    conn = adata_ref.uns["paga"]["connectivities"]
    if sp.issparse(conn):
        conn = conn.toarray()
    paga_df = pd.DataFrame(
        conn,
        index=adata_ref.obs[reference_label_col].cat.categories,
        columns=adata_ref.obs[reference_label_col].cat.categories
    )
    paga_df.to_csv(os.path.join(outdir, "BTD10_reference_PAGA_connectivity.csv"))


## 6. Load Xenium query data

In [ ]:

# ============================================================
# 9) Load Xenium spatial object
# ============================================================
xen = sc.read_h5ad(xenium_h5ad_path)
xen = ensure_cell_id_columns(xen)
xen = ensure_spatial_columns(xen)
xen = make_unique_gene_names_upper(xen)

print(xen)
print("Xenium obs columns:", xen.obs.columns.tolist()[:30])


In [ ]:

# ============================================================
# 10) Optional: subset to an ROI using a Selection1-style cell list
# ============================================================
if roi_stats_csv_path is not None and os.path.exists(roi_stats_csv_path):
    roi = read_csv_flex(roi_stats_csv_path)
    if "Cell ID" not in roi.columns:
        raise ValueError("ROI stats file must contain 'Cell ID'.")
    roi_ids = roi["Cell ID"].astype(str).str.strip()
    roi_ids_clean = roi_ids.str.replace("-1$", "", regex=True)

    roi_mask = (
        xen.obs["cell_id_raw"].astype(str).isin(roi_ids) |
        xen.obs["cell_id_clean"].astype(str).isin(roi_ids_clean)
    )
    xen.obs["ROI_selected"] = np.where(roi_mask, "ROI", "Other")
    xen = xen[roi_mask].copy()
    print("Xenium cells retained in ROI:", xen.n_obs)
else:
    xen.obs["ROI_selected"] = "ROI"
    print("No ROI filter applied.")


## 7. Optional external annotation for Xenium cells

In [ ]:

# ============================================================
# 11) Optional external annotation file
# ============================================================
if use_for_xenium_csv is not None and os.path.exists(use_for_xenium_csv):
    anno = read_csv_flex(use_for_xenium_csv)

    rename_map = {}
    for c in anno.columns:
        cl = c.strip().lower()
        if cl in ["cell_id", "cellid", "cell id"]:
            rename_map[c] = "cell_id"
        elif cl in ["group", "celltype", "cell_type", "label"]:
            rename_map[c] = "group"
    anno = anno.rename(columns=rename_map)

    if "cell_id" not in anno.columns or "group" not in anno.columns:
        raise ValueError("Annotation CSV must contain cell_id and group columns.")

    anno["cell_id"] = anno["cell_id"].astype(str).str.strip()
    anno["group"] = anno["group"].astype(str).str.strip()
    anno = anno.drop_duplicates("cell_id")

    cell_to_group = dict(zip(anno["cell_id"], anno["group"]))

    mapped_raw = xen.obs["cell_id_raw"].map(cell_to_group)
    mapped_clean = xen.obs["cell_id_clean"].map(cell_to_group)
    mapped = mapped_raw.where(~mapped_raw.isna(), mapped_clean)

    xen.obs[transferred_label_col] = mapped.astype("object")
    print("Externally annotated Xenium cells:", int((~mapped.isna()).sum()))
else:
    print("No external annotation file used.")


## 8. Transfer velocity-informed states from reference to Xenium

In [ ]:

# ============================================================
# 12) Transfer labels and velocity-derived metrics to Xenium
# ============================================================
xen = transfer_reference_to_query(
    adata_ref,
    xen,
    label_col=reference_label_col,
    metrics=("velocity_pseudotime", "velocity_confidence", "velocity_length"),
    n_pcs=n_pcs,
    k=n_neighbors
)

# Use external Xenium labels if present; otherwise use transferred reference labels
if transferred_label_col not in xen.obs.columns or xen.obs[transferred_label_col].isna().all():
    xen.obs[transferred_label_col] = xen.obs[reference_label_col + "_transfer"].astype(str)

xen.obs[transferred_label_col] = xen.obs[transferred_label_col].astype(str)
xen.obs[spatial_group_col] = xen.obs[transferred_label_col].astype("category")

print(xen.obs[[transferred_label_col, "transfer_confidence", "velocity_pseudotime", "velocity_confidence"]].head())


In [ ]:

# ============================================================
# 13) Save transferred spatial object
# ============================================================
xen.write(os.path.join(outdir, "Xenium_with_transferred_velocity_states.h5ad"))
print("Saved:", os.path.join(outdir, "Xenium_with_transferred_velocity_states.h5ad"))


## 9. Spatial plots

In [ ]:

# ============================================================
# 14) Spatial plots in Xenium coordinates
# ============================================================
fig, ax = plt.subplots(figsize=(10, 10))
sc.pl.scatter(xen, basis="spatial", color=transferred_label_col, size=12, ax=ax, show=False)
ax.set_title("Transferred cell states in Xenium space")
plt.tight_layout()
plt.savefig(os.path.join(outdir, "Xenium_transferred_labels_spatial.png"), dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

fig, ax = plt.subplots(figsize=(10, 10))
scv.pl.scatter(xen, basis="spatial", color="velocity_pseudotime", color_map="plasma", size=15, ax=ax, show=False)
ax.set_title("Transferred velocity pseudotime in Xenium space")
plt.tight_layout()
plt.savefig(os.path.join(outdir, "Xenium_velocity_pseudotime_spatial.png"), dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

fig, ax = plt.subplots(figsize=(10, 10))
scv.pl.scatter(xen, basis="spatial", color="velocity_confidence", color_map="viridis", size=15, ax=ax, show=False)
ax.set_title("Transferred velocity confidence in Xenium space")
plt.tight_layout()
plt.savefig(os.path.join(outdir, "Xenium_velocity_confidence_spatial.png"), dpi=300, bbox_inches="tight", facecolor="white")
plt.show()


## 10. H&E overlay

In [ ]:

# ============================================================
# 15) H&E overlays
# ============================================================
if he_image_path is not None and os.path.exists(he_image_path):
    plot_he_overlay(
        xen,
        color_col=transferred_label_col,
        image_path=he_image_path,
        outpath=os.path.join(outdir, "Xenium_transferred_labels_on_HE.png"),
        preferred_x=he_x_col,
        preferred_y=he_y_col,
        point_size=12
    )

    plot_he_overlay(
        xen,
        color_col="velocity_pseudotime",
        image_path=he_image_path,
        outpath=os.path.join(outdir, "Xenium_velocity_pseudotime_on_HE.png"),
        preferred_x=he_x_col,
        preferred_y=he_y_col,
        point_size=12
    )

    plot_he_overlay(
        xen,
        color_col="velocity_confidence",
        image_path=he_image_path,
        outpath=os.path.join(outdir, "Xenium_velocity_confidence_on_HE.png"),
        preferred_x=he_x_col,
        preferred_y=he_y_col,
        point_size=12
    )
else:
    print("H&E image not provided. Skipping overlay plots.")


## 11. Standard PAGA on the Xenium query (no manual arrows)

In [ ]:

# ============================================================
# 16) PAGA on Xenium using transferred or external labels
# ============================================================
xen_for_paga = xen.copy()

# Use transferred PCA if present, otherwise compute PCA on normalized expression
if "X_ref_pca" in xen_for_paga.obsm.keys():
    xen_for_paga.obsm["X_pca"] = xen_for_paga.obsm["X_ref_pca"]
else:
    sc.pp.normalize_total(xen_for_paga, target_sum=1e4)
    sc.pp.log1p(xen_for_paga)
    sc.pp.pca(xen_for_paga, n_comps=min(n_pcs, xen_for_paga.n_obs - 1))

sc.pp.neighbors(xen_for_paga, n_neighbors=min(n_neighbors, xen_for_paga.n_obs - 1), use_rep="X_pca")
sc.tl.umap(xen_for_paga)
sc.tl.paga(xen_for_paga, groups=spatial_group_col)

sc.pl.paga(xen_for_paga, color=spatial_group_col, threshold=0.03)
sc.pl.paga_compare(
    xen_for_paga,
    basis="umap",
    color=spatial_group_col,
    threshold=0.03,
    right_margin=0.2,
    size=20
)

if "paga" in xen_for_paga.uns:
    conn = xen_for_paga.uns["paga"]["connectivities"]
    if sp.issparse(conn):
        conn = conn.toarray()
    paga_df = pd.DataFrame(
        conn,
        index=xen_for_paga.obs[spatial_group_col].cat.categories,
        columns=xen_for_paga.obs[spatial_group_col].cat.categories
    )
    paga_df.to_csv(os.path.join(outdir, "Xenium_PAGA_connectivity.csv"))


## 12. Optional summary tables

In [ ]:

# ============================================================
# 17) Summary tables
# ============================================================
summary = (
    xen.obs
    .groupby(transferred_label_col)[["velocity_pseudotime", "velocity_confidence", "velocity_length", "transfer_confidence"]]
    .describe()
)
summary.to_csv(os.path.join(outdir, "Xenium_velocity_summary_by_group.csv"))

xen.obs[[
    "cell_id_raw",
    "cell_id_clean",
    transferred_label_col,
    "velocity_pseudotime",
    "velocity_confidence",
    "velocity_length",
    "transfer_confidence",
    "x_centroid",
    "y_centroid"
]].to_csv(os.path.join(outdir, "Xenium_per_cell_velocity_outputs.csv"))

print("Saved summary tables.")



## Notes

- The notebook uses **standard scVelo dynamical RNA velocity** for the scRNA-seq reference.
- It uses **standard Scanpy PAGA** for graph abstraction.
- It **does not include manual PAGA arrows**.
- The spatial transfer step is implemented as a **shared-gene, PCA-based nearest-neighbor transfer** of labels and velocity-derived metrics from the reference to the Xenium query.
- If you have a project-specific SIRV implementation, you can replace the transfer step with your original model call and keep the downstream plotting sections unchanged.



## Suggested software

You can document the workflow with software such as:

- Python
- Scanpy
- scVelo
- AnnData
- Velocyto
- Matplotlib
- tifffile
- scikit-learn

Add exact versions from your environment before posting to GitHub or using the notebook for a manuscript supplement.
